# Environment Diagnostics

Run this notebook to verify your environment is ready for the course.

**What it checks:**
1. Python version and platform
2. Core package imports (FloPy, GeoPandas, etc.)
3. Core geospatial operations (GeoPandas, Shapely, pyproj) and optional extra imports
4. Visualization libraries (matplotlib, plotly, folium)
5. MODFLOW executable (auto-downloads if missing)

**When to run:**
- Once at the start of the course
- After changing your Python environment
- If you encounter unexpected errors

**Expected result:** `overall_ready = True` in the final summary.

## 1. Setup

In [19]:
from pathlib import Path
import sys

# Add support module path
support_src = Path('_SUPPORT/src').resolve()
if str(support_src) not in sys.path:
    sys.path.insert(0, str(support_src))

# Initialize results container
diag_results = {
    'python': {},
    'packages': {},
    'geospatial': {},
    'viz': {},
    'modflow': {},
    'system': {}
}
print('Ready to run diagnostics.')

Ready to run diagnostics.


## 2. Python & Platform

In [20]:
import platform
import datetime

py_info = {
    'python_version': sys.version.split()[0],
    'platform': platform.platform(),
    'checked_at': datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
}
diag_results['python'] = py_info

print(f"Python: {py_info['python_version']}")
print(f"Platform: {py_info['platform']}")

Python: 3.13.2
Platform: Windows-11-10.0.26300-SP0


## 3. Core Package Imports

Checking that essential packages can be imported.

In [21]:
CORE_PACKAGES = [
    ('numpy', 'np'),
    ('pandas', 'pd'),
    ('matplotlib', None),
    ('flopy', None),
    ('geopandas', 'gpd'),
    ('shapely', None),
    ('rasterio', None),
    ('pyproj', None),
    ('folium', None),
    ('plotly', None),
    ('ipywidgets', None),
]

pkg_status = {}
missing = []

for pkg, alias in CORE_PACKAGES:
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', 'unknown')
        pkg_status[pkg] = {'ok': True, 'version': version}
        print(f"  ✓ {pkg} ({version})")
    except ImportError as e:
        pkg_status[pkg] = {'ok': False, 'error': str(e)}
        missing.append(pkg)
        print(f"  ✗ {pkg} - MISSING")

diag_results['packages'] = {'status': pkg_status, 'missing': missing}

if missing:
    print(f"\n⚠ Missing packages: {', '.join(missing)}")
    print("Run 'uv sync' to install dependencies.")
else:
    print("\n✓ All core packages available.")

  ✓ numpy (2.3.5)
  ✓ pandas (2.2.3)
  ✓ matplotlib (3.10.1)
  ✓ flopy (3.9.5)
  ✓ geopandas (1.2.0)
  ✓ shapely (2.2.0)
  ✓ rasterio (1.4.3)
  ✓ pyproj (3.7.1)
  ✓ folium (1.0.0)
  ✓ plotly (7.1.0)
  ✓ ipywidgets (8.1.9)

✓ All core packages available.


## 3.5 — Flow-calibration dependency (pyemu)

The flow calibration notebooks (05f–08f) require **pyemu** for PEST++ parameter estimation.
This section auto-installs pyemu into the running kernel if the hub image does not already
include it (one-time, ~seconds). The 2026 hub image ships pyemu, so this becomes a
harmless no-op once the image is updated.

In [22]:
from env_utils import ensure_package
pyemu_status = ensure_package('pyemu', 'pyemu==1.4.0')
diag_results['pyemu'] = pyemu_status
_msg = {'present': '✓ pyemu already installed',
        'installed': '✓ pyemu installed now',
        'installed_needs_restart': '⚠ pyemu installed — RESTART THE KERNEL and re-run',
        'failed': '✗ pyemu install FAILED'}
print(_msg.get(pyemu_status['status'], pyemu_status['status']),
      pyemu_status.get('version', pyemu_status.get('error', '')))

✓ pyemu already installed 1.4.0


## 3.6 — Pinned versions (`uv.lock`)

A few packages decide **model numbers**, not just convenience. If the versions installed here
differ from the ones the project pins, results can differ from the reference outputs even though
nothing looks broken.

This **reports only — it installs nothing.** The JupyterHub image is not something you can change
from a notebook, and upgrading a foundational package in a shared kernel would risk an environment
inconsistent with the image. If this section reports drift, **tell the course staff** — it is a
deployment fix, not a student one.


In [23]:
from env_utils import ensure_pinned_versions, ensure_security_floors

# The JupyterHub base image ships OLDER numpy/flopy than this project pins, and the
# image cannot be modified. numpy and flopy set the bit pattern of the frozen
# reference outputs, so a drifted kernel cannot tell a real regression from an
# environment difference -- the checks report "inconclusive", never a silent pass.
# This tops the packages up into your USER site; it does not touch the shared image.
pin_report = ensure_pinned_versions()

if not pin_report["lock_found"]:
    print("  ? uv.lock not found - cannot verify pinned versions")
else:
    for name, row in pin_report["packages"].items():
        inst, want, ok = row["installed"], row["locked"], row["matches"]
        action = pin_report["actions"].get(name, {}).get("status")
        if ok is True:
            print(f"  \u2713 {name} {inst}")
        elif action == "installed_needs_restart":
            print(f"  \u2191 {name} {inst} -> {want} installed (restart needed)")
        elif action == "failed":
            err = pin_report["actions"][name].get("error", "")
            print(f"  \u2717 {name} {inst} - could not install {want}: {err[:120]}")
        elif ok is False:
            print(f"  \u2717 {name} {inst}  (project pins {want})")
        else:
            print(f"  ? {name} installed={inst} locked={want} - cannot compare")

if pin_report["needs_restart"]:
    print()
    print("  \u26a0 RESTART THE KERNEL and re-run this notebook.")
    print("    The new versions are installed but the running kernel still holds the")
    print("    old ones, so nothing you run before restarting uses them.")

_still_wrong = [n for n, a in pin_report["actions"].items() if a["status"] == "failed"]
if _still_wrong:
    print()
    print("  \u26a0 Version drift remains: " + ", ".join(_still_wrong))
    print("    These packages decide model numbers. Your results may differ from the")
    print("    reference outputs. Report this to the course staff.")

diag_results['pinned_versions'] = pin_report


# --- Security floors -------------------------------------------------------
# Some packages carry security advisories. We want them at a MINIMUM version
# (>=), never at an exact one: pinning exactly would DOWNGRADE a Hub whose image
# already runs ahead of this project's lock. The floor is read from uv.lock, so
# keeping it current needs no Hub-admin request.
#
# Packages belonging to the Jupyter SERVER's own stack (e.g. tornado) are only
# REPORTED here, never installed automatically: a broken copy in your user site
# can stop your server from starting, and you would have no notebook left to fix
# it from. Course staff handle those with the Hub admins.
sec_report = ensure_security_floors()

if not sec_report["lock_found"]:
    print("  ? uv.lock not found - cannot verify security floors")
else:
    for name, row in sec_report["packages"].items():
        inst, floor = row["installed"], row["floor"]
        status = sec_report["actions"].get(name, {}).get("status")
        if status == "ok":
            print(f"  \u2713 {name} {inst} (floor {floor})")
        elif status == "installed_needs_restart":
            print(f"  \u2191 {name} {inst} -> >={floor} installed (restart needed)")
        elif status == "server_stack_reported":
            print(f"  \u26a0 {name} {inst} is below the {floor} security floor.")
            print("    NOT installed on purpose - it is part of the Jupyter server.")
            print("    Please tell the course staff; do not install it yourself.")
        elif status == "failed":
            err = sec_report["actions"][name].get("error", "")
            print(f"  \u2717 {name} {inst} - could not reach >={floor}: {err[:120]}")
        else:
            print(f"  ? {name} installed={inst} floor={floor} - {status}")

if sec_report["needs_restart"]:
    print()
    print("  \u26a0 RESTART THE KERNEL and re-run this notebook.")
    print("    To undo a user-site install:  pip uninstall -y <package>")

diag_results['security_floors'] = sec_report


  ✓ numpy 2.3.5
  ✓ scipy 1.17.0
  ✓ flopy 3.9.5
  ⚠ tornado 6.4.2 is below the 6.6 security floor.
    NOT installed on purpose - it is part of the Jupyter server.
    Please tell the course staff; do not install it yourself.
  ✓ anyio 4.16.0 (floor 4.15.1)
  ✓ soupsieve 4.1.3 (floor 2.9.2)


## 4. Geospatial Stack

Readiness requires successful GeoPandas, Shapely, and pyproj operations and a valid union.
Optional extras (fiona, rasterio, contextily) are informational: student notebooks do not
require them, and no action is needed if they are missing. Core failures should be
reported to course staff; optional probes are skipped after a core failure.

In [24]:
import diagnostics

geo_checks = diagnostics.geospatial_smoke_test()
diag_results['geospatial'] = geo_checks

if 'error' in geo_checks:
    print(f"✗ Geospatial error: {geo_checks['error']} - report to course staff")
else:
    print("✓ GeoPandas/Shapely working")
    print(f"  Created {geo_checks.get('polygons_count', '?')} polygons, union valid: {geo_checks.get('union_valid', '?')}")
    
for lib in ['fiona', 'rasterio', 'contextily']:
    status = geo_checks.get(lib)
    if status is True:
        print(f"✓ {lib}")
    elif status:
        print(f"ℹ Optional {lib}: {status}. No action needed; student notebooks do not require it.")

✓ GeoPandas/Shapely working
  Created 2 polygons, union valid: True
ℹ Optional fiona: ERROR: No module named 'fiona'. No action needed; student notebooks do not require it.
✓ rasterio
ℹ Optional contextily: ERROR: No module named 'contextily'. No action needed; student notebooks do not require it.


## 5. Visualization

In [25]:
viz_status = diagnostics.viz_smoke_test()
diag_results['viz'] = viz_status

for lib in ['matplotlib', 'plotly', 'folium']:
    status = viz_status.get(lib, 'not checked')
    if 'ok' in str(status).lower():
        print(f"✓ {lib}")
    else:
        print(f"✗ {lib}: {status}")

✓ matplotlib
✓ plotly
✓ folium


## 6. MODFLOW Executable

This will attempt to auto-download MODFLOW if not found.

In [26]:
PERSISTENT_WORKSPACE = Path.home() / 'applied_groundwater_modelling_data' / 'diagnostics'
PERSISTENT_WORKSPACE.mkdir(parents=True, exist_ok=True)

modflow_diag = diagnostics.modflow_minimal_model(
    auto_download=True,
    persistent_workspace=PERSISTENT_WORKSPACE,
    cleanup_on_success=False,
)

diag_results['modflow'] = modflow_diag

if modflow_diag.get('executable_found'):
    print(f"✓ MODFLOW found: {modflow_diag.get('executable_path', 'unknown')}")
else:
    print("✗ MODFLOW not found")
    if modflow_diag.get('attempted_download'):
        print(f"  Download attempted but failed. Try manually: get-modflow :flopy")

if modflow_diag.get('run_success'):
    print("✓ Test model ran successfully")
    if modflow_diag.get('analytical_ok'):
        print("✓ Results match analytical solution")
elif modflow_diag.get('executable_found'):
    print(f"✗ Test model failed: {modflow_diag.get('error', 'unknown error')}")

✓ MODFLOW found: C:\Users\shado\AppData\Local\flopy\bin\mf2005.EXE
✓ Test model ran successfully
✓ Results match analytical solution


## 6.5 Data Cache (one-time downloads)

Pre-downloads the course's large GIS files (~70 MB, dominated by the 46 MB DEM and 23 MB groundwater map, plus small README sidecars) so notebook 04f doesn't pause mid-run to fetch them. Idempotent — already-present files are skipped, so re-running is cheap. Set `RUN_DATA_PREFETCH = False` below to skip (files then download on first use).

In [27]:
import os  # ensure os is in scope for this cell

RUN_DATA_PREFETCH = True   # set False to skip; files download on first use in the notebooks
if RUN_DATA_PREFETCH:
    from data_utils import warm_data_cache, format_cache_report
    cache_report = warm_data_cache(verbose=True)
else:
    cache_report = {"items": [], "counts": {}, "total_mb": 0, "ok": None}
    print("Data prefetch skipped (RUN_DATA_PREFETCH = False). Files download on first use.")
diag_results['data_cache'] = cache_report

Note: config.py not found — using config_template.py (default for public data).


limmat_model_boundary.gpkg:   0%|          | 0.00/96.0k [00:00<?, ?iB/s]

Download complete: C:\Users\shado\applied_groundwater_modelling_data\limmat\gis\limmat_model_boundary.gpkg


limmat_model_boundary_readme.md:   0%|          | 0.00/591 [00:00<?, ?iB/s]

Download complete: C:\Users\shado\applied_groundwater_modelling_data\limmat\gis\limmat_model_boundary_readme.md


limmat_model_boundary_segments.gpkg:   0%|          | 0.00/104k [00:00<?, ?iB/s]

... (15 outputs left out)

swissalti3d_merged_lv95.tif:   0%|          | 0.00/45.8M [00:00<?, ?iB/s]

Download complete: C:\Users\shado\applied_groundwater_modelling_data\limmat\gis\swissalti3d_merged_lv95.tif


swissalti3d_merged_lv95_readme.pdf:   0%|          | 0.00/1.65M [00:00<?, ?iB/s]

Download complete: C:\Users\shado\applied_groundwater_modelling_data\limmat\gis\swissalti3d_merged_lv95_readme.pdf
Data cache prefetch — GIS files (~70 MB; actual usage slightly higher due to README sidecars)
Name                           Status        Size (MB)
--------------------------------------------------------
  ✓  model_boundary               DOWNLOADED          0.1
  ✓  model_boundary_segments      DOWNLOADED          0.1
  ✓  wells                        DOWNLOADED          0.6
  ✓  rivers                       DOWNLOADED          5.0
  ✓  groundwater_map_norm         DOWNLOADED         23.2
  ✓  dem_hres                     DOWNLOADED         45.8
--------------------------------------------------------
  0 cached / 6 downloaded / 0 failed / 75 MB total


### Calibrated flow model — is it the shipped one?

The case-study notebooks are pinned to frozen reference grids that were built on the **shipped**
calibrated flow model. If this workspace holds a *different* calibration — for example because
`05f_calibration.ipynb` was run locally with `RUN_PEST_LOCALLY=True`, which regenerates it in
place — then those builds will refuse to run, reporting that the grid diverged from its golden.

Unlike a version mismatch, **you can fix this one yourself**: remove the directory and let it
re-download.


In [28]:
import model_io_utils as _mio

try:
    _flow_ws = _mio.ensure_flow_model()
    flow_report = _mio.verify_flow_model(_flow_ws)
    if flow_report["is_canonical"]:
        print(f"  \u2713 calibrated flow model is the shipped one ({flow_report['fingerprint']})")
    else:
        print(f"  \u2717 calibrated flow model is NOT the shipped one")
        print(f"      here    : {flow_report['fingerprint']}")
        print(f"      shipped : {flow_report['canonical']}")
        print(f"      Case-study builds pinned to the frozen grids will REFUSE this model.")
        print(f"      To restore it:  mv {_flow_ws} {_flow_ws}.local-backup")
        print(f"      then re-run this notebook - it will re-download the shipped archive.")
    if not flow_report["manifest_consistent"]:
        print("  \u2717 the workspace does not match its own MANIFEST_flow.json "
              "(partial extraction?)")
except Exception as _exc:
    flow_report = {"error": f"{type(_exc).__name__}: {_exc}"}
    print(f"  ? calibrated flow model unavailable: {_exc}")

diag_results['flow_model'] = flow_report


limmat_valley_calibrated_flow_model.zip:   0%|          | 0.00/1.08M [00:00<?, ?iB/s]

Download complete: C:\Users\shado\applied_groundwater_modelling_data\limmat\calibration\limmat_valley_calibrated_flow_model.zip


limmat_valley_calibrated_flow_model_readme.md:   0%|          | 0.00/1.86k [00:00<?, ?iB/s]

Download complete: C:\Users\shado\applied_groundwater_modelling_data\limmat\calibration\limmat_valley_calibrated_flow_model_readme.md
  ✓ calibrated flow model is the shipped one (6a9e27c455dcbb66)


## 7. Summary

Environment readiness requires essential packages, a working core geospatial stack,
and a MODFLOW executable and successful test run. Missing or incomplete core geospatial
results block readiness; missing optional extras do not. The separate calibrated flow
model check also blocks readiness when the model is non-canonical.

In [29]:
summary = diagnostics.build_summary(diag_results)
diag_results['summary'] = summary

print('=' * 40)
print('DIAGNOSTICS SUMMARY')
print('=' * 40)

checks = [
    ('Core packages', not summary.get('missing_essential')),
    ('Geospatial stack', summary.get('geospatial_success', False)),
    ('MODFLOW executable', summary.get('modflow_executable_found')),
    ('MODFLOW test run', summary.get('modflow_run_success')),
]

for name, passed in checks:
    status = '✓' if passed else '✗'
    print(f"  {status} {name}")

if not summary.get('geospatial_success', False):
    error = diag_results.get('geospatial', {}).get('error', 'Core success was not recorded')
    print(f"  ✗ Geospatial core: {error} - report to course staff")
if summary.get('geospatial_optional_errors'):
    extras = ', '.join(summary['geospatial_optional_errors'])
    print(f"  ℹ Optional geospatial extras unavailable: {extras}. "
          "No action needed; student notebooks do not require them.")

# Non-gating data cache status (network state does not affect overall_ready)
dc = diag_results.get('data_cache', {}); c = dc.get('counts', {})
if dc.get('ok') is None:
    print("  - Data cache: skipped")
else:
    print(f"  {'✓' if dc.get('ok') else '⚠'} Data cache: {c.get('CACHED',0)} cached, "
          f"{c.get('DOWNLOADED',0)} downloaded, {c.get('FAILED',0)} failed "
          f"({dc.get('total_mb',0):.0f} MB)")

# Version drift: NON-GATING (⚠). A student cannot change the JupyterHub image, and a
# red they cannot action trains them to ignore reds. It is evidence for course staff.
pv = diag_results.get('pinned_versions', {})
if not pv.get('lock_found'):
    print("  - Pinned versions: uv.lock not found")
elif pv.get('mismatches'):
    print(f"  ⚠ Pinned versions: {', '.join(pv['mismatches'])} differ from uv.lock "
          f"- report to course staff")
else:
    print("  ✓ Pinned versions match uv.lock")

# Calibrated flow model: GATING (✗). Unlike version drift this DOES invalidate outputs -
# case-study builds pinned to the frozen grids refuse a different calibration - and
# unlike the image, the student CAN fix it.
fm = diag_results.get('flow_model', {})
if fm.get('error'):
    print(f"  - Calibrated flow model: unavailable ({fm['error'][:40]})")
elif fm.get('is_canonical'):
    print("  ✓ Calibrated flow model is the shipped one")
else:
    print("  ✗ Calibrated flow model is NOT the shipped one - case-study builds will refuse it")

# Non-gating pyemu status (does not affect overall_ready)
pe = diag_results.get('pyemu', {})
_ok = pe.get('status') in ('present', 'installed')
print(f"  {'✓' if _ok else '⚠'} pyemu: {pe.get('status','?')}"
      + (f" ({pe.get('version')})" if pe.get('version') else ''))

print('=' * 40)
# A non-canonical calibration is a REAL blocker for the case studies, so it gates.
_flow_blocks = (fm.get('is_canonical') is False)
if summary.get('overall_ready') and not _flow_blocks:
    print('\n✓ ENVIRONMENT READY')
    print('  You can proceed with the course materials.')
else:
    print('\n✗ ENVIRONMENT NOT READY')
    print('  Review the checks above and fix any issues.')
    if summary.get('missing_essential'):
        print(f"  Missing packages: {summary['missing_essential']}")
    if _flow_blocks:
        print('  The calibrated flow model is not the shipped one - see section 6.5 '
              'above for the one-line fix.')

DIAGNOSTICS SUMMARY
  ✓ Core packages
  ✓ Geospatial stack
  ✓ MODFLOW executable
  ✓ MODFLOW test run
  ℹ Optional geospatial extras unavailable: fiona, contextily. No action needed; student notebooks do not require them.
  ✓ Data cache: 0 cached, 6 downloaded, 0 failed (75 MB)
  ✓ Pinned versions match uv.lock
  ✓ Calibrated flow model is the shipped one
  ✓ pyemu: present (1.4.0)

✓ ENVIRONMENT READY
  You can proceed with the course materials.
